In [1]:
# How has quality of life changed in Poland - downloading Eurostat data for Tableau
# Requirements: pip install eurostat pandas openpyxl
import eurostat
import pandas as pd

In [2]:
EU27 = ["AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "EL", "ES", "FI", "FR", "HR",
        "HU", "IE", "IT", "LT", "LU", "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK",
        "EU27_2020"]

COUNTRY_NAMES = {
    "AT": "Austria",     "BE": "Belgium",      "BG": "Bulgaria",   "CY": "Cyprus",      "CZ": "Czechia",
    "DE": "Germany",     "DK": "Denmark",      "EE": "Estonia",    "EL": "Greece",      "ES": "Spain",
    "FI": "Finland",     "FR": "France",       "HR": "Croatia",    "HU": "Hungary",     "IE": "Ireland",
    "IT": "Italy",       "LT": "Lithuania",    "LU": "Luxembourg", "LV": "Latvia",      "MT": "Malta",
    "NL": "Netherlands", "PL": "Poland",       "PT": "Portugal",   "RO": "Romania",     "SE": "Sweden",
    "SI": "Slovenia",    "SK": "Slovakia",     "EU27_2020": "EU27 (average)",
}

# column name: (Eurostat dataset, filters, startPeriod)
DATASETS = {
    "gdp_pc_pps":         ("nama_10_pc",    {"unit": "CP_PPS_EU27_2020_HAB", "na_item": "B1GQ"}, 2000),
    "min_wage_pps":       ("earn_mw_cur",   {"currency": "PPS"}, "2000-S1"),
    "life_expectancy":    ("demo_mlexpec",  {"unit": "YR", "sex": "T", "age": "Y_LT1"}, 2000),
    "unemployment":       ("une_rt_a",      {"unit": "PC_ACT", "age": "Y15-74", "sex": "T"}, 2000),
    "homicides_per_100k": ("crim_hom_soff", {"unit": "P_HTHAB", "iccs": "ICCS0101", "leg_stat": "PER_VICT", "sex": "T"}, 2000),
    "arop_60":            ("ilc_li02",      {"statinfo": "MED_EI", "unit": "PC", "rskpovth": "B_60", "sex": "T", "age": "TOTAL"}, 2005),
    "pop_0_14":           ("demo_pjanbroad", {"unit": "NR", "sex": "T", "age": "Y_LT15"}, 1995),
    "pop_15_64":          ("demo_pjanbroad", {"unit": "NR", "sex": "T", "age": "Y15-64"}, 1995),
    "pop_65_plus":        ("demo_pjanbroad", {"unit": "NR", "sex": "T", "age": "Y_GE65"}, 1995),
}


def fetch_indicator(code, filters, start):
    allowed = set(eurostat.get_par_values(code, "geo"))
    geo = [g for g in EU27 if g in allowed]
    missing = sorted(set(EU27) - allowed)
    if missing:
        print(f"{code}: codes not available in the dataset: {missing}")

    df = eurostat.get_data_df(code, filter_pars={"geo": geo, "startPeriod": start, **filters})
    geo_col = next(c for c in df.columns if str(c).startswith("geo"))
    year_columns = [c for c in df.columns if str(c)[:4].isdigit()]

    d = (df.rename(columns={geo_col: "geo"})
           .melt(id_vars="geo", value_vars=year_columns, var_name="period", value_name="value")
           .dropna(subset=["value"]))
    d["period"] = d["period"].astype(str)
    d["year"] = d["period"].str[:4].astype(int)

    # after applying the filters, each country must have exactly one value per period
    assert not d.duplicated(["geo", "period"]).any(), f"{code}: narrow down the filters"

    # semi-annual data (minimum wage) -> yearly average; everything else unchanged
    return d.groupby(["geo", "year"], as_index=False)["value"].mean()


# --- download and merge all indicators ---
table = None
for name, (code, filters, start) in DATASETS.items():
    d = fetch_indicator(code, filters, start).rename(columns={"value": name})
    table = d if table is None else table.merge(d, on=["geo", "year"], how="outer")

table.insert(1, "country", table["geo"].map(COUNTRY_NAMES))
table = table.sort_values(["geo", "year"]).reset_index(drop=True)

print(table.info())

# data coverage for Poland (first and last year with a value)
pl = table[table["geo"] == "PL"]
for indicator in DATASETS:
    years = pl.loc[pl[indicator].notna(), "year"]
    print(f"{indicator}: {years.min()}-{years.max()}" if len(years) else f"{indicator}: no data for PL")

# --- export ---
table.to_excel("dane_ue.xlsx", index=False)



earn_mw_cur: codes not available in the dataset: ['EU27_2020']
crim_hom_soff: codes not available in the dataset: ['EU27_2020']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 880 entries, 0 to 879
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   geo                 880 non-null    object 
 1   country             880 non-null    object 
 2   year                880 non-null    int64  
 3   gdp_pc_pps          728 non-null    float64
 4   min_wage_pps        548 non-null    float64
 5   life_expectancy     695 non-null    float64
 6   unemployment        486 non-null    float64
 7   homicides_per_100k  402 non-null    float64
 8   arop_60             572 non-null    float64
 9   pop_0_14            856 non-null    float64
 10  pop_15_64           856 non-null    float64
 11  pop_65_plus         856 non-null    float64
dtypes: float64(9), int64(1), object(2)
memory usage: 82.6+ KB
None
gdp_pc_pps: 2